# Lite-DPAS 실험 노트북

DPAS의 공간·지연·스펙트럼 처리 아이디어를 경량 구조로 옮긴 **실험용 모델**입니다. 이 노트북은 설정 확인 → 모델 동작 확인 → 학습 → validation 프로파일 순서로 진행합니다.

> 기본 실행은 모델 동작 확인만 하고 학습은 시작하지 않습니다. 짧은 학습은 설정 셀에서 `RUN_TRAINING = True`를 켜고 1 epoch로 실행하세요. 전체 학습은 `EPOCHS = 100`으로 바꿉니다. 학습과 프로파일링 모두 test split은 사용하지 않습니다.

## 모델 구성

- **Dual-axis lite block:** depthwise 3×3 공간 경로와 depthwise 1×5 지연 경로를 1×1 convolution으로 융합
- **Learned multiscale analysis:** 학습형 stride-2 블록으로 32×32 → 16×16 → 8×8 특징을 만듦
- **Compact latent:** 8×8×16 특징 맵을 선택한 실수 latent 차원으로 압축
- **Learned synthesis:** PixelShuffle 업샘플링으로 8×8 → 16×16 → 32×32 복원
- **Band-wise spectral gate:** 복원 특징을 지연축 FFT로 변환해 4개 대역별로 조절
- **Output:** 정규화된 실수부·허수부 CSI, shape `[B, 2, 32, 32]`

In [ ]:
from pathlib import Path
import json, sys, time
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

# 노트북 위치와 현재 작업 폴더 양쪽에서 프로젝트 루트를 찾습니다.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir() and (PROJECT_ROOT / 'CSI_SLLM_RESEARCH' / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT / 'CSI_SLLM_RESEARCH'
if not (PROJECT_ROOT / 'src').is_dir():
    raise FileNotFoundError('CSI_SLLM_RESEARCH 프로젝트 폴더를 찾지 못했습니다. PROJECT_ROOT를 직접 지정하세요.')
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / 'Data'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Project:', PROJECT_ROOT)
print('Data:', DATA_DIR)
print('Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '')

In [ ]:
# 실험 설정: 첫 실행은 1 epoch로 확인한 다음 전체 학습으로 바꾸세요.
COMPRESSION_RATIO = 4       # 지원값: 4, 16, 32, 64
EPOCHS = 1                  # 본 학습은 예: 100
BATCH_SIZE = 200            # GPU 메모리 부족 시 64 또는 32
SEED = 42
RUN_TAG = 'v2_try_01'       # 재실험은 v2_try_02 등 새 이름 사용
RUN_TRAINING = False        # 학습을 시작하려면 True로 변경
PROFILE_SAMPLES = 512      # validation 프로파일에 사용할 표본 수

assert COMPRESSION_RATIO in (4, 16, 32, 64)
ENCODED_DIM = 2048 // COMPRESSION_RATIO
required = ['DATA_Htrainin.mat', 'DATA_Hvalin.mat']
missing = [name for name in required if not (DATA_DIR / name).is_file()]
assert not missing, f'Data 폴더에 파일이 없습니다: {missing}'
print(f'CR=1/{COMPRESSION_RATIO} | latent={ENCODED_DIM} real values | epochs={EPOCHS} | batch={BATCH_SIZE}')
print('Train/validation 파일 확인 완료. Test 파일은 이 노트북에서 열지 않습니다.')

## 1. 모델 동작 확인

무작위 입력으로 shape, 출력 범위, 역전파를 확인합니다. 이 단계는 학습 결과 평가가 아닙니다.

In [ ]:
from src.models.lite_dpas import LiteDPASCsiNetV2, count_parameters
from src.models.csinet import CsiNet

model = LiteDPASCsiNetV2(encoded_dim=ENCODED_DIM).to(DEVICE)
sample = torch.rand(2, 2, 32, 32, device=DEVICE)
output = model(sample)
assert output.shape == sample.shape
assert torch.isfinite(output).all()
assert output.min() >= 0 and output.max() <= 1
((output - sample) ** 2).mean().backward()

baseline_params = count_parameters(CsiNet(encoded_dim=ENCODED_DIM))
lite_params = count_parameters(model)
print('Forward/backward: OK')
print('Input/output:', tuple(sample.shape), tuple(output.shape))
print(f'Parameters: Lite-DPAS {lite_params:,} | CsiNet port {baseline_params:,}')
print(f'Parameter change: {(1 - lite_params / baseline_params) * 100:.1f}% fewer than this CsiNet port')

## 2. 학습

기본값 `RUN_TRAINING = False`에서는 학습을 시작하지 않습니다. 먼저 1 epoch를 실행해 checkpoint 저장과 validation 선택이 잘 되는지 확인하세요. 실행 경로는 `runs/`이며 기존 체크포인트와 이름이 겹치면 덮어쓸 수 있으니, 재학습 전에 보관 여부를 확인하세요.

In [ ]:
from src.train_baseline import train_model

if RUN_TRAINING:
    CHECKPOINT = train_model(
        'litedpas_v2', PROJECT_ROOT, DATA_DIR,
        epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED, encoded_dim=ENCODED_DIM, run_tag=RUN_TAG,
    )
    print('Best validation checkpoint:', CHECKPOINT)
else:
    CHECKPOINT = PROJECT_ROOT / 'runs' / f'litedpas_v2_indoor_cr{COMPRESSION_RATIO}_{EPOCHS}ep_{RUN_TAG}' / 'best.pt'
    print('학습은 실행하지 않았습니다. 설정 셀에서 RUN_TRAINING=True로 바꾸세요.')
    print('예상 checkpoint:', CHECKPOINT)

## 3. Validation 성능·비용 프로파일

학습 checkpoint를 불러와 validation 일부의 NMSE, 파라미터 수, Conv/Linear MAC 추정치, batch-1 지연을 기록합니다. MAC 추정치는 FFT·활성화·정규화·보간 비용을 제외하므로 실제 지연 측정과 함께 해석하세요. 실행 횟수가 적은 노트북 수치는 장치 상태에 영향을 받을 수 있습니다.

In [ ]:
from src.cost2100 import COST2100Dataset
from src.metrics import sample_nmse_linear
from src.profile_lite_dpas import approximate_macs, measure_latency_ms

if not CHECKPOINT.is_file():
    print('Checkpoint가 없습니다. 먼저 학습 셀을 실행하세요.')
else:
    saved = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
    profiled_model = LiteDPASCsiNetV2(encoded_dim=int(saved.get('encoded_dim', ENCODED_DIM)))
    profiled_model.load_state_dict(saved['model'])
    profiled_model = profiled_model.to(DEVICE).eval()
    validation = COST2100Dataset(DATA_DIR / 'DATA_Hvalin.mat')
    n = min(PROFILE_SAMPLES, len(validation))
    val_loader = DataLoader(Subset(validation, range(n)), batch_size=64, shuffle=False, num_workers=0)
    ratios = []
    with torch.inference_mode():
        for batch in val_loader:
            batch = batch.to(DEVICE)
            ratios.append(sample_nmse_linear(batch, profiled_model(batch)).cpu())
    val_nmse_db = float(10 * np.log10(max(torch.cat(ratios).mean().item(), 1e-12)))
    one = torch.zeros(1, 2, 32, 32, device=DEVICE)
    latency = measure_latency_ms(profiled_model, one, warmup=20, repeats=100)
    macs = approximate_macs(profiled_model, one)
    report = {
        'model': 'Lite-DPAS-V2', 'compression_ratio': f"1/{2048 // int(saved.get('encoded_dim', ENCODED_DIM))}",
        'validation_samples': n, 'validation_nmse_db': val_nmse_db,
        'parameters': count_parameters(profiled_model),
        'approx_conv_linear_macs_batch1': macs,
        'batch1_latency_ms': latency, 'device': str(DEVICE), 'test_split_used': False,
        'mac_note': 'FFT, activation, normalization and interpolation excluded',
    }
    display(report)
    report_path = CHECKPOINT.parent / 'profile_validation.json'
    report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
    print('Saved:', report_path)

## 다음 실험 순서

1. V2의 learned down/up-sampling이 동작하는지 forward/backward 확인
2. CR=1/4에서 1 epoch 학습으로 실행 경로 점검
3. 같은 설정으로 100 epoch 학습 후 validation NMSE 확인
4. CsiNet과 동일 seed·데이터·학습 조건으로 비교
5. 설정을 고정한 뒤 CR=1/16, 1/32, 1/64와 outdoor로 확장
6. 공간 경로, 지연 경로, spectral gate를 하나씩 제거하는 ablation 수행

여기서 얻은 validation 결과는 모델 선택 자료입니다. 최종 성능을 주장하기 전에는 설정을 고정하고 별도 최종 평가 절차에서 test split을 사용하세요.